In [ ]:
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm
import numpy as np
from difflib import SequenceMatcher
import os

# 평가 metrics

In [ ]:
def calculate_char_accuracy(pred, target):
    """문자 단위 정확도"""
    if len(target) == 0:
        return 0.0
    correct = sum(1 for p, t in zip(pred, target) if p == t)
    return correct / max(len(pred), len(target))

def calculate_word_accuracy(pred, target):
    """단어 단위 정확도"""
    pred_words = pred.split()
    target_words = target.split()
    if len(target_words) == 0:
        return 0.0
    correct = sum(1 for p, t in zip(pred_words, target_words) if p == t)
    return correct / max(len(pred_words), len(target_words))

def calculate_similarity(pred, target):
    """문자열 유사도 (SequenceMatcher)"""
    return SequenceMatcher(None, pred, target).ratio()

# data load

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
print("데이터 로딩 중...")
train = pd.read_csv('/content/drive/MyDrive/data/open/train.csv', encoding = 'utf-8-sig')
test = pd.read_csv('/content/drive/MyDrive/data/open/test.csv', encoding = 'utf-8-sig')

데이터 로딩 중...


In [ ]:
# 평가용 샘플 (전체 사용 시 시간이 오래 걸릴 수 있음)
train_samples = train[:200]  # Few-shot 예시용
eval_samples = train[200:300]  # 평가용 (train의 일부를 평가용으로)

In [ ]:
# Few-shot 예시 생성
samples = []
for i in range(10):
    sample = f"input : {train_samples['input'].iloc[i]} \n output : {train_samples['output'].iloc[i]}"
    samples.append(sample)

In [ ]:
# 평가할 모델 리스트
models_to_evaluate = [
    "beomi/gemma-ko-2b",
    # 추가 모델들을 여기에 넣으세요
]

In [ ]:
# 결과 파일 경로
results_file_path = '/content/drive/MyDrive/data/open/model_evaluation_results.csv'

# 기존 결과 파일 로드
already_evaluated = []
if os.path.exists(results_file_path):
    print("\n기존 평가 결과 파일을 찾았습니다.")
    existing_results = pd.read_csv(results_file_path, encoding='utf-8-sig')
    already_evaluated = existing_results['model_name'].tolist()
    print(f"이미 평가된 모델: {already_evaluated}")

def save_result_immediately(result_dict, results_file_path):
    """각 모델 평가 후 즉시 결과를 파일에 저장"""
    if os.path.exists(results_file_path):
        existing = pd.read_csv(results_file_path, encoding='utf-8-sig')
        existing = existing[existing['model_name'] != result_dict['model_name']]
        updated = pd.concat([existing, pd.DataFrame([result_dict])], ignore_index=True)
    else:
        updated = pd.DataFrame([result_dict])

    updated = updated.sort_values('avg_score', ascending=False).reset_index(drop=True)
    updated.to_csv(results_file_path, index=False, encoding='utf-8-sig')
    print(f"✓ 결과가 저장되었습니다: {results_file_path}")


기존 평가 결과 파일을 찾았습니다.
이미 평가된 모델: ['paust/pko-t5-base', 'lcw99/t5-base-korean-text-summary', 'psyche/KoT5-summarization', 'cosmoquester/bart-ko-small', 'eenzeenee/t5-base-korean-summarization', 'gogamza/kobart-summarization', 'Jo-2j/Dacon-Encrypted', 'gogamza/kobart-base-v2', 'facebook/mbart-large-50', 'KETI-AIR/ke-t5-base', 'cchyun/nmt-koen-t5-small', 'lcw99/t5-base-korean-grammar-correction', 'MLP-KTLim/llama-3-Korean-Bllossom-AICA-3B', nan, nan, nan, nan, nan]


In [ ]:
# 평가 결과 저장
results = []

for model_name in models_to_evaluate:
    if model_name in already_evaluated:
        print(f"\n{'='*60}")
        print(f"모델 '{model_name}'은 이미 평가되었습니다. 건너뜁니다.")
        print(f"{'='*60}\n")
        continue

    print(f"\n{'='*60}")
    print(f"모델 평가 중: {model_name}")
    print(f"{'='*60}\n")

    try:
        # 모델 및 토크나이저 로드
        tokenizer = AutoTokenizer.from_pretrained(model_name)

        # pad_token 설정
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,  # 메모리 절약
            device_map="auto",
            low_cpu_mem_usage=True
        )

        model.eval()

        # 평가 메트릭 저장
        char_accuracies = []
        word_accuracies = []
        similarities = []

        # 각 샘플에 대해 추론
        for idx, row in tqdm(eval_samples.iterrows(), total=len(eval_samples), desc=f"Evaluating {model_name}"):
            query = row['input']
            target = row['output']

            # CausalLM용 프롬프트 (더 간결하게)
            prompt = f"""다음은 난독화된 한글 리뷰를 복원하는 예시입니다:

{samples[0]}

{samples[1]}

이제 다음 난독화된 리뷰를 복원하세요:
난독화: {query}
복원:"""

            try:
                inputs = tokenizer(
                    prompt,
                    return_tensors="pt",
                    max_length=1024,  # CausalLM은 긴 컨텍스트 처리 가능
                    truncation=True
                )

                # token_type_ids 제거
                if 'token_type_ids' in inputs:
                    inputs.pop('token_type_ids')

                inputs = {k: v.to(model.device) for k, v in inputs.items()}

                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=100,  # 새로 생성할 토큰 수만 지정
                        do_sample=True,
                        temperature=0.7,
                        top_p=0.9,
                        pad_token_id=tokenizer.pad_token_id,
                        eos_token_id=tokenizer.eos_token_id,
                        num_return_sequences=1
                    )

                generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

                # 프롬프트 제거하여 순수 출력만 추출
                if "복원:" in generated_text:
                    # 마지막 "복원:" 이후의 텍스트만 추출
                    result = generated_text.split("복원:")[-1].strip()
                    # 다음 줄이나 특수 문자로 끝나는 경우 처리
                    result = result.split("\n")[0].strip()
                else:
                    result = generated_text[len(prompt):].strip()
                    result = result.split("\n")[0].strip()

                # 메트릭 계산
                char_acc = calculate_char_accuracy(result, target)
                word_acc = calculate_word_accuracy(result, target)
                sim = calculate_similarity(result, target)

                char_accuracies.append(char_acc)
                word_accuracies.append(word_acc)
                similarities.append(sim)

            except Exception as e:
                print(f"Error processing sample {idx}: {str(e)}")
                continue

        # 평균 메트릭 계산
        avg_char_acc = np.mean(char_accuracies) if char_accuracies else 0.0
        avg_word_acc = np.mean(word_accuracies) if word_accuracies else 0.0
        avg_sim = np.mean(similarities) if similarities else 0.0

        # 결과 저장
        result_dict = {
            "model_name": model_name,
            "char_accuracy": avg_char_acc,
            "word_accuracy": avg_word_acc,
            "similarity": avg_sim,
            "avg_score": (avg_char_acc + avg_word_acc + avg_sim) / 3,
            "model_type": "CausalLM"
        }
        results.append(result_dict)

        # 즉시 파일에 저장
        save_result_immediately(result_dict, results_file_path)

        print(f"\n[{model_name}] 평가 결과:")
        print(f"  - 문자 정확도: {avg_char_acc:.4f}")
        print(f"  - 단어 정확도: {avg_word_acc:.4f}")
        print(f"  - 유사도: {avg_sim:.4f}")
        print(f"  - 평균 점수: {result_dict['avg_score']:.4f}")

        # 메모리 및 디스크 정리
        del model
        del tokenizer
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"Error loading model {model_name}: {str(e)}")
        error_result = {
            "model_name": model_name,
            "char_accuracy": 0.0,
            "word_accuracy": 0.0,
            "similarity": 0.0,
            "avg_score": 0.0,
            "model_type": "CausalLM",
            "error": str(e)
        }
        results.append(error_result)
        save_result_immediately(error_result, results_file_path)
        continue


모델 평가 중: beomi/gemma-ko-2b



Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating beomi/gemma-ko-2b: 100%|██████████| 100/100 [04:31<00:00,  2.71s/it]

✓ 결과가 저장되었습니다: /content/drive/MyDrive/data/open/model_evaluation_results.csv

[beomi/gemma-ko-2b] 평가 결과:
  - 문자 정확도: 0.1375
  - 단어 정확도: 0.0203
  - 유사도: 0.2529
  - 평균 점수: 0.1369


In [ ]:
# 결과 파일 경로
results_file_path = '/content/drive/MyDrive/data/open/model_evaluation_results.csv'

# 기존 결과 파일이 있으면 로드
import os
if os.path.exists(results_file_path):
    print("\n기존 평가 결과 파일을 찾았습니다. 결과를 추가합니다...")
    existing_results = pd.read_csv(results_file_path, encoding='utf-8-sig')

    # 새로운 결과를 데이터프레임으로 변환
    new_results_df = pd.DataFrame(results)

    # 기존 결과와 병합 (중복 모델은 새 결과로 업데이트)
    combined_results = pd.concat([existing_results, new_results_df], ignore_index=True)
    combined_results = combined_results.drop_duplicates(subset=['model_name'], keep='last')
    results_df = combined_results.sort_values('avg_score', ascending=False).reset_index(drop=True)
else:
    print("\n새로운 평가 결과 파일을 생성합니다...")
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('avg_score', ascending=False).reset_index(drop=True)


기존 평가 결과 파일을 찾았습니다. 결과를 추가합니다...


In [ ]:
print("\n" + "="*60)
print("최종 평가 결과 (성능 순)")
print("="*60)
print(results_df.to_string(index=False))


최종 평가 결과 (성능 순)
                               model_name  char_accuracy  word_accuracy  similarity  avg_score                                                                                                                                                                                                                                                                                                                         error model_type
                        beomi/gemma-ko-2b       0.137509       0.020325    0.252868   0.136901                                                                                                                                                                                                                                                                                                                           NaN   CausalLM
                        paust/pko-t5-base       0.044480       0.000000    0.153325   0.065935                                         

In [ ]:
# 결과 저장
results_df.to_csv(results_file_path, index=False, encoding='utf-8-sig')
print(f"\n평가 결과가 '{results_file_path}'에 저장되었습니다.")
print(f"총 {len(results_df)}개의 모델이 평가되었습니다.")